# c06：阿里云 OSS Markdown 入库 Elasticsearch（生产流程）

本笔记本执行 **真实** 流程：列举 OSS → **入库前审查 ES 中将被替换的 chunk** → 可选按 `source_path` **精确范围删除** → 下载 → **第 9 节分步**：切分 → 核对 → BGE → bulk。

- **教学 / 短样例验证**（`chunk_version`、虚拟数据）见 [c05_ingest_es_runbook.ipynb](c05_ingest_es_runbook.ipynb)。
- **不删错**：删除与预览使用 **同一** `source_path` 列表（`terms` 查询），不碰其它路径下的 chunk。
- **不漏删**：先删尽本范围内旧 `_id`（同一文件下多余 `chunk_index` 一并删除），再 bulk 新块。

**依赖**：`uv sync --extra oss --extra embedding --extra es`；`.env` 配置 `OSS_*`、`ES_*`、`BGE_M3_PATH`、`MODEL_*`。


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)

# 显式加载项目根 .env，保证 OSS SDK 读取到 AK/SK
from dotenv import load_dotenv

load_dotenv(_ROOT / ".env", override=False)

print("仓库根:", _ROOT)


仓库根: /Users/zheng/projects/python/ai/rag/rag_law


## 0. 依赖（OSS SDK / torch / elasticsearch）


In [6]:
!uv sync --extra oss --extra embedding

Resolved 176 packages in 6ms
Installed 52 packages in 381ms                              
 + accelerate==1.13.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + attrs==26.1.0
 + beautifulsoup4==4.14.3
 + cbor==1.0.0
 + datasets==4.8.4
 + dill==0.4.1
 + filelock==3.25.2
 + flagembedding==1.3.5
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + hf-xet==1.4.3
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + inscriptis==2.7.1
 + ir-datasets==0.5.11
 + joblib==1.5.3
 + lxml==6.0.2
 + lz4==4.4.5
 + mpmath==1.3.0
 + multidict==6.7.1
 + multiprocess==0.70.19
 + networkx==3.6.1
 + pandas==3.0.2
 + peft==0.18.1
 + pillow==12.2.0
 + propcache==0.4.1
 + protobuf==7.34.1
 + pyarrow==23.0.1
 + regex==2026.4.4
 + safetensors==0.7.0
 + scikit-learn==1.8.0
 + scipy==1.17.1
 + sentence-transformers==3.4.1
 + sentencepiece==0.2.1
 + setuptools==81.0.0
 + soupsieve==2.8.3
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.2
 + torch==2.11.0
 + tqdm==4.67.3
 + transformers==4.57.6
 + tr

In [7]:
import importlib.util

if importlib.util.find_spec("alibabacloud_oss_v2") is None:
    raise RuntimeError("请执行: uv sync --extra oss")
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("请执行: uv sync --extra embedding")
import elasticsearch  # noqa: F401
print("依赖 OK")


依赖 OK


## 1. 参数配置

- **TARGET_INDEX**：默认 `ES_INDEX + "_c06"`；若写入生产索引，将 `USE_PRODUCTION_INDEX` 设为 `True`（**慎用**）。
- **LIMIT**：`None` 为前缀下全部 `.md`；设为正整数用于抽样。
- **PRE_DELETE_SCOPED**：`True` 时在 bulk 前按 `source_path` 列表删除 ES 中旧文档（与预览一致）。


In [24]:
from conf.settings import get_settings

get_settings.cache_clear()
settings = get_settings()

# 与 scripts/ingest_oss_md_to_es.py 保持一致，优先使用 Settings(.env)
OSS_BUCKET = settings.oss_bucket
OSS_PREFIX = settings.oss_object_prefix
DOWNLOAD_DIR = Path(settings.oss_download_dir)
if not DOWNLOAD_DIR.is_absolute():
    DOWNLOAD_DIR = (_ROOT / DOWNLOAD_DIR).resolve()

LIMIT = None  # 或 3 用于试跑

USE_PRODUCTION_INDEX = False
TARGET_INDEX = settings.es_index if USE_PRODUCTION_INDEX else (settings.es_index + "")

CHUNK_VERSION = (settings.chunk_version or "").strip() or "1.1.7"
PRE_DELETE_SCOPED = True
# 这次临时跳过 source_path 严格判断（不做 assert 中断）
SKIP_SOURCE_PATH_CHECK = True

region_cfg = (settings.oss_region or os.getenv("OSS_REGION") or "").strip()
assert region_cfg, "请在 .env 中设置 OSS_REGION（如北京为 cn-beijing）"

print("OSS_BUCKET =", repr(OSS_BUCKET))
print("OSS_PREFIX =", repr(OSS_PREFIX))
print("OSS_REGION =", repr(region_cfg))
print("DOWNLOAD_DIR =", DOWNLOAD_DIR)
print("TARGET_INDEX =", TARGET_INDEX, "| CHUNK_VERSION =", CHUNK_VERSION)
print("PRE_DELETE_SCOPED =", PRE_DELETE_SCOPED, "| LIMIT =", LIMIT)
print("SKIP_SOURCE_PATH_CHECK =", SKIP_SOURCE_PATH_CHECK)


OSS_BUCKET = 'rag-law'
OSS_PREFIX = 'md3/'
OSS_REGION = 'cn-beijing'
DOWNLOAD_DIR = /Users/zheng/projects/python/ai/rag/rag_law/data/md_minerU
TARGET_INDEX = rag_law_chunks_c06 | CHUNK_VERSION = 1.1.7
PRE_DELETE_SCOPED = True | LIMIT = None
SKIP_SOURCE_PATH_CHECK = True


## 2. 列举 OSS 下待处理的 .md keys


In [9]:
from utils.oss_aliyun.crud import list_all_object_keys


def select_md_keys(all_keys: list[str], limit: int | None) -> list[str]:
    md_sorted = sorted(k for k in all_keys if k.lower().endswith(".md"))
    if limit is None:
        return md_sorted
    return md_sorted[: int(limit)]

all_keys = list_all_object_keys(OSS_BUCKET, prefix=OSS_PREFIX)
md_keys = select_md_keys(all_keys, LIMIT)
print("本批 .md 对象数:", len(md_keys))
for k in md_keys[:50]:
    print(" ", k)
if len(md_keys) > 50:
    print(" ... 共", len(md_keys), "个")
assert md_keys, "无 .md 对象，请检查桶/前缀/权限"


本批 .md 对象数: 8
  md3/刑法.md
  md3/劳动合同法.md
  md3/劳动法.md
  md3/婚姻法.md
  md3/宪法.md
  md3/民事诉讼法.md
  md3/民法典.md
  md3/行政复议法.md


## 3. 计算与 `ingest_oss_md_to_es.py` 一致的 `source_path` / 本地路径


In [10]:
from ingest.oss_layout import iter_oss_md_ingest_plan

plan = iter_oss_md_ingest_plan(_ROOT, DOWNLOAD_DIR, md_keys)
SOURCE_PATHS = [t[0] for t in plan]

for sp, lp, ok in plan[:20]:
    print(sp, "<=", ok)
print("source_path 总数:", len(SOURCE_PATHS))
assert len(set(SOURCE_PATHS)) == len(SOURCE_PATHS), "source_path 应唯一"


data/md_minerU/md3__刑法.md <= md3/刑法.md
data/md_minerU/md3__劳动合同法.md <= md3/劳动合同法.md
data/md_minerU/md3__劳动法.md <= md3/劳动法.md
data/md_minerU/md3__婚姻法.md <= md3/婚姻法.md
data/md_minerU/md3__宪法.md <= md3/宪法.md
data/md_minerU/md3__民事诉讼法.md <= md3/民事诉讼法.md
data/md_minerU/md3__民法典.md <= md3/民法典.md
data/md_minerU/md3__行政复议法.md <= md3/行政复议法.md
source_path 总数: 8


## 4. 连接 ES 并 ping


In [11]:
from es_store.client import elasticsearch_client

client = elasticsearch_client(settings)
assert client.ping(), "ES 不可达"
print("ping OK")


ping OK


## 5. **入库前审查**：将被本批覆盖的 chunk（索引不存在时为空）

与「按范围删除」使用相同 `terms`：`source_path` = 上一节列表。


In [25]:
from es_store.scope_replace import preview_chunks_for_source_paths

print("—— 索引:", TARGET_INDEX)
if client.indices.exists(index=TARGET_INDEX):
    preview = preview_chunks_for_source_paths(client, TARGET_INDEX, SOURCE_PATHS, sample_size=40)
    print("—— 命中总数（若执行 PRE_DELETE 将删除的条数）:", preview["total"])
    print("—— 按 source_path 聚合:")
    for row in preview["per_path"]:
        print(
            " ",
            row["source_path"],
            "doc_count=",
            row["doc_count"],
            "max_chunk_index=",
            row["max_chunk_index"],
        )
    print("—— 抽样:")
    for s in preview["samples"]:
        print(s)
else:
    print("索引", TARGET_INDEX, "尚不存在，预览为空（首次写入）。")


—— 索引: rag_law_chunks_c06
索引 rag_law_chunks_c06 尚不存在，预览为空（首次写入）。


## 6. 人工确认

将 `CONFIRM` 改为 `True` 表示已核对第 5 节，同意在 **TARGET_INDEX** 上删除上述范围内的旧文档并写入新数据。


In [26]:
CONFIRM = True  # 改为 True 后继续

assert CONFIRM is True, "请确认后设 CONFIRM=True"


## 7. 按 `source_path` 精确删除旧 chunk（`PRE_DELETE_SCOPED=True`）

删除后校验范围内命中数为 **0**。


In [27]:
from es_store.scope_replace import (
    delete_chunks_by_source_paths,
    verify_no_chunks_for_source_paths,
)

if PRE_DELETE_SCOPED:
    if client.indices.exists(index=TARGET_INDEX):
        resp = delete_chunks_by_source_paths(client, TARGET_INDEX, SOURCE_PATHS)
        print("delete_by_query:", {k: resp.get(k) for k in ("deleted", "failures") if k in resp})
        left = verify_no_chunks_for_source_paths(client, TARGET_INDEX, SOURCE_PATHS)
        print("删除后范围内剩余命中数:", left)
        if not SKIP_SOURCE_PATH_CHECK:
            assert left == 0
        else:
            print("(已跳过 source_path 严格断言)")
    else:
        print("索引不存在，跳过删除。")
else:
    print("已跳过范围删除（PRE_DELETE_SCOPED=False）。")


索引不存在，跳过删除。


## 8. 从 OSS 下载到本地


In [15]:
from utils.oss_aliyun.crud import download_object_to_file
from utils.oss_aliyun.public_urls import oss_virtual_host_public_url

region = (settings.oss_region or os.getenv("OSS_REGION") or "").strip()
assert region, "请设置 OSS_REGION"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
url_by_source_path: dict[str, str] = {}
md_paths: list[Path] = []

for sp, local, key in plan:
    download_object_to_file(OSS_BUCKET, key, local)
    url_by_source_path[sp] = oss_virtual_host_public_url(OSS_BUCKET, region, key)
    md_paths.append(local)

print("已下载", len(md_paths), "个文件")


已下载 8 个文件


## 9. 切分 → 向量 → 建索引 → bulk（**分步执行**）

以下拆成多个单元格，建议顺序：

1. **9.0** 准备语义合并用的 embedder（若 `CHUNK_SEMANTIC_MERGE_SIMILARITY=embedding`）并打印当前 `.env` 切分相关配置。
2. **9.1** 仅执行 **切分**，得到 `chunks` / `shas`。
3. **9.2** **按 `source_path` 逐文件**打印每块的 `chunk_index`、`char_start`/`char_end`、长度与正文预览（用于人工验收）。
4. **9.3** 向量编码（确认切分无误后再跑）。
5. **9.4** `ensure_index`（索引已存在则跳过创建）。
6. **9.5** 组装 `docs` 并 **bulk** 写入。

切分参数与 [`chunking.webui`](../../src/chunking/webui) 一致，来自 **`.env`** / `Settings`。


### 9.0 准备（语义合并 embedding 时需先建 embedder）

In [28]:
from embeddings import build_embedder

merge_embedder = None
if (
    settings.chunk_semantic_merge_enabled
    and settings.chunk_semantic_merge_similarity == "embedding"
):
    merge_embedder = build_embedder(settings)

print(
    "切分配置: CHUNK_SIZE=",
    settings.chunk_size,
    "CHUNK_OVERLAP=",
    settings.chunk_overlap,
    "CHUNK_BOUNDARY_AWARE=",
    settings.chunk_boundary_aware,
    "CHUNK_SEMANTIC_MERGE_ENABLED=",
    settings.chunk_semantic_merge_enabled,
)


切分配置: CHUNK_SIZE= 1500 CHUNK_OVERLAP= 100 CHUNK_BOUNDARY_AWARE= True CHUNK_SEMANTIC_MERGE_ENABLED= False


### 9.1 切分（仅本步：得到 `chunks` / `shas`）

In [17]:
from ingest import load_chunks_with_sha256

chunks, shas = load_chunks_with_sha256(
    None,
    md_paths=md_paths,
    boundary_aware=settings.chunk_boundary_aware,
    semantic_merge_enabled=settings.chunk_semantic_merge_enabled,
    semantic_merge_similarity=settings.chunk_semantic_merge_similarity,
    embedding_backend=merge_embedder,
    semantic_merge_threshold=settings.chunk_semantic_merge_threshold,
    semantic_merge_min_chars=settings.chunk_semantic_merge_min_chars,
    semantic_merge_max_chars=settings.chunk_semantic_merge_max_chars,
)

assert len(chunks) == len(shas)
print("总 chunk 条数:", len(chunks))


总 chunk 条数: 233


### 9.2 按文档核对切分结果（每文件下列出各块元数据与正文预览）

可按需修改 `TEXT_PREVIEW_CHARS`（单块正文预览最大字符数）。


In [18]:
from collections import defaultdict

TEXT_PREVIEW_CHARS = 400

by_path: dict[str, list] = defaultdict(list)
for c, sha in zip(chunks, shas):
    sp = c.source_path or c.source_file or "?"
    by_path[sp].append((c, sha))

for sp in sorted(by_path.keys()):
    items = by_path[sp]
    print("=" * 72)
    print("source_path:", sp)
    print("  块数:", len(items), "| 示例 SHA256 前缀:", items[0][1][:16], "...")
    for c, sha in items:
        t = c.text or ""
        prev = t[:TEXT_PREVIEW_CHARS]
        if len(t) > TEXT_PREVIEW_CHARS:
            prev = prev + "\n  …（正文已截断，共 %s 字）" % len(t)
        print(
            "  [chunk %s] char[%s:%s) len(text)=%s"
            % (c.chunk_index, c.char_start, c.char_end, len(t))
        )
        for line in prev.splitlines():
            print("   ", line)
    print()


source_path: data/md_minerU/md3__刑法.md
  块数: 56 | 示例 SHA256 前缀: 5ddf490da306b864 ...
  [chunk 0] char[0:1472) len(text)=1472
    # 中华人民共和国刑法
    
    第一编 总则
    
    第一章 刑法的任务、基本原则和适用范围
    
    第二章 犯罪
    
    第一节 犯罪和刑事责任
    
    第二节 犯罪的预备、未遂和中止
    
    第三节 共同犯罪
    
    第四节 单位犯罪
    
    第三章 刑罚
    
    第一节 刑罚的种类
    
    第二节 管制
    
    第三节 拘役
    
    第四节 有期徒刑、无期徒刑
    
    第五节 死刑
    
    第六节 罚金
    
    第七节 剥夺政治权利
    
    第八节 没收财产
    
    第四章 刑罚的具体运用
    
    第一节 量刑
    
    第二节 累犯
    
    第三节 自首和立功
    
    第四节 数罪并罚
    
    第五节 缓刑
    
    第六节 减刑
    
    第七节 假释
    
    第八节 时效
    
    第五章 其他规定
    
    第二编 分则
    
    第一章 危害国家安全罪
    
    第二章 危害公共安全罪
    
    第三章 破坏社会主义市场经济秩序罪
    
    第一节 生产、销售伪劣商品罪
    
    第二节 走私罪
    
    第三节 妨害对公司、企业的管理秩序罪
    
    第四节 破坏金融管理秩序罪
    
    第五节 金融诈
      …（正文已截断，共 1472 字）
  [chunk 1] char[1397:2918) len(text)=1521
    
    
    第九条 【普遍管辖权】对于中华人民共和国缔结或者参加的国际条约所规定的罪行，中华人民共和国在所承担条约义务的范围内行使刑事管辖权的，适用本法。
    
    第十条 【对外国刑事判决的消极承认】凡在中华人民共和国领

### 9.2b 导出切分结果到 `data/chunk_md`

将当前 `chunks` 按 **源文档**（`source_path`）分组，同文件内各块按 `chunk_index` 排序，块与块之间用 **三个换行**（`\n\n\n`）拼接，写入 `data/chunk_md/<原文件名>.md`（仅文件名，扁平存放）。

便于离线 diff 或与 chunk 预览对照；**不影响**后续 9.3 向量步骤。


In [21]:
from collections import defaultdict

CHUNK_MD_DIR = _ROOT / "data" / "chunk_md"
CHUNK_MD_DIR.mkdir(parents=True, exist_ok=True)

by_sp: dict[str, list] = defaultdict(list)
for c in chunks:
    sp = c.source_path or c.source_file or "unknown"
    by_sp[sp].append(c)

for sp in sorted(by_sp.keys()):
    parts = sorted(by_sp[sp], key=lambda x: x.chunk_index)
    symbol = "\n\n"+"-"*200+"\n\n"
    body = symbol.join((p.text or "") for p in parts)
    name = Path(sp).name
    out_path = CHUNK_MD_DIR / name
    out_path.write_text(body, encoding="utf-8")
    print("写入", out_path.relative_to(_ROOT), "块数:", len(parts))

print("完成，目录:", CHUNK_MD_DIR)


写入 data/chunk_md/md3__刑法.md 块数: 56
写入 data/chunk_md/md3__劳动合同法.md 块数: 9
写入 data/chunk_md/md3__劳动法.md 块数: 7
写入 data/chunk_md/md3__婚姻法.md 块数: 4
写入 data/chunk_md/md3__宪法.md 块数: 14
写入 data/chunk_md/md3__民事诉讼法.md 块数: 26
写入 data/chunk_md/md3__民法典.md 块数: 107
写入 data/chunk_md/md3__行政复议法.md 块数: 10
完成，目录: /Users/zheng/projects/python/ai/rag/rag_law/data/chunk_md


### 9.3 向量编码（确认 9.2 无误后再执行）

In [22]:
from embeddings import build_embedder

embedder = merge_embedder if merge_embedder is not None else build_embedder(settings)
embeddings = embedder.embed_documents([c.text for c in chunks])
assert len(embeddings) == len(chunks)
dim = embedder.dense_dimension
print("向量条数:", len(embeddings), "dense 维度:", dim)


/Users/zheng/projects/python/ai/rag/rag_law/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 124.88it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:03<00:00,  3.53s/it]

向量条数: 233 dense 维度: 1024


### 9.4 确保目标索引存在（`recreate=False`，不删已有索引）

In [29]:
from es_store.store import EsChunkStore

store = EsChunkStore(client, TARGET_INDEX, dense_dims=dim)
store.ensure_index(recreate=False)
print("索引就绪:", TARGET_INDEX)


索引就绪: rag_law_chunks_c06


### 9.5 组装文档并 bulk 写入

In [30]:
from ingest.documents import chunk_embedding_to_source

docs = []
for c, emb, sha in zip(chunks, embeddings, shas):
    sp = c.source_path or ""
    oss_u = url_by_source_path.get(sp, "")
    docs.append(
        chunk_embedding_to_source(
            c,
            emb,
            source_sha256=sha,
            source_oss_url=oss_u,
            chunk_version=CHUNK_VERSION,
        )
    )

ok_n, errs = store.bulk_index_chunks(docs)
assert not errs, errs
assert ok_n == len(docs)
store.refresh()
print("bulk 成功:", ok_n, "维度:", dim)


bulk 成功: 233 维度: 1024


## 10. **入库后验证**：条数与 `chunk_version`


In [31]:
from es_store.scope_replace import preview_chunks_for_source_paths

post = preview_chunks_for_source_paths(client, TARGET_INDEX, SOURCE_PATHS, sample_size=min(50, len(docs)))
print("范围内总命中:", post["total"], "| 本批 chunk 数:", len(docs))
if not SKIP_SOURCE_PATH_CHECK:
    assert post["total"] == len(docs)
else:
    print("(已跳过 source_path 总量严格断言)")

for h in client.search(
    index=TARGET_INDEX,
    size=min(30, len(docs)),
    query={"terms": {"source_path": SOURCE_PATHS}},
    sort=[{"source_path": "asc"}, {"chunk_index": "asc"}],
)["hits"]["hits"]:
    assert h["_source"].get("chunk_version") == CHUNK_VERSION

print("验证通过")


范围内总命中: 233 | 本批 chunk 数: 233
(已跳过 source_path 总量严格断言)
验证通过
